In [2]:
import pandas as pd
from pathlib import Path

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d chrisfilo/urbansound8k

!mkdir /content/urbansound8k
!unzip -q urbansound8k.zip -d /content/urbansound8k

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/chrisfilo/urbansound8k
License(s): Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)
100% 5.61G/5.61G [01:15<00:00, 79.8MB/s]



In [4]:
download_path = '/content/urbansound8k'

metadata_file = download_path + '/UrbanSound8K.csv'
df = pd.read_csv(metadata_file)
df.head()

df['relative_path'] = '/fold' + df['fold'].astype(str) + '/' + df['slice_file_name'].astype(str)

df = df[['relative_path', 'classID']]
df.head()

,relative_path,classID
0,/fold5/100032-3-0-0.wav,3
1,/fold5/100263-2-0-117.wav,2
2,/fold5/100263-2-0-121.wav,2
3,/fold5/100263-2-0-126.wav,2
4,/fold5/100263-2-0-137.wav,2


In [5]:
import math, random
import torch
import torchaudio
from torchaudio import transforms
from IPython.display import Audio
import torch.nn.functional as F
import soundfile as sf

In [ ]:
class AudioUtil():
    @staticmethod
    def open(audio_file):
        data, sr = sf.read(audio_file, dtype='float32')

        sig = torch.from_numpy(data)

        if sig.ndim == 1:
            sig = sig.unsqueeze(0)
        else:
            sig = sig.permute(1, 0)

        return (sig, sr)
    @staticmethod
    def rechannel(aud, new_channel):
        sig, sr = aud
        if sig.shape[0] == new_channel:
            return aud
        if new_channel == 1:
            resig = sig[:1, :]
        else:
            resig = torch.cat([sig, sig])
        return (resig, sr)

    @staticmethod
    def resample(aud, newsr):
        sig, sr = aud
        if sr == newsr:
            return aud
        sig, sr = aud
        num_channels = sig.shape[0]
        resig = torchaudio.transforms.Resample(sr, newsr)(sig[:1, :])
        if num_channels > 1:
            retwo = torchaudio.transforms.Resample(sr, newsr)(sig[1:, :])
            resig = torch.cat([resig, retwo])
        return (resig, newsr)

    @staticmethod
    def pad_trunc(aud, max_ms):
        sig, sr = aud
        num_rows, sig_len = sig.shape
        max_len = sr*max_ms//1000
        if sig_len == max_len:
            return (sig, sr)
        if sig_len > max_len:
            sig = sig[:, :max_len]
        else:
            sig = F.pad(sig, (0, max_len - sig_len), mode='constant', value=0.0)
        return (sig, sr)

    @staticmethod
    def time_shift(aud, shift_limit):
        sig, sr = aud
        _, sig_len = sig.shape
        shift_amt = int(random.random() * shift_limit * sig_len)
        return (sig.roll(shift_amt), sr)

    @staticmethod
    def spectro_gram(aud, n_mels, n_fft=1024, hop_len=None):
        sig, sr = aud
        top_db = 80
        spec = transforms.MelSpectrogram(sr, n_fft=n_fft, hop_length=hop_len, n_mels=n_mels)(sig)
        spec = transforms.AmplitudeToDB(top_db=top_db)(spec)
        return (spec)

    @staticmethod
    def spectro_augment(spec, max_mask_pct=0.1, n_freq_masks=1, n_time_masks=1):
        _, n_mels, n_steps = spec.shape
        mask_value = spec.mean()
        aug_spec = spec
        freq_mask_param = max_mask_pct * n_mels
        for _ in range(n_freq_masks):
            aug_spec = transforms.FrequencyMasking(freq_mask_param)(aug_spec, mask_value)

        time_mask_param = max_mask_pct * n_steps
        for _ in range(n_time_masks):
            aug_spec = transforms.TimeMasking(time_mask_param)(aug_spec, mask_value)
        return aug_spec


In [7]:
from torch.utils.data import DataLoader, Dataset, random_split

In [8]:
class SoundDS(Dataset):
    def __init__(self, df, data_path):
        self.df = df
        self.data_path = str(data_path)
        self.duration = 4000
        self.sr = 44100
        self.channel = 2
        self.shift_pct = 0.4

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        audio_file = self.data_path + self.df.loc[idx, 'relative_path']
        class_id = self.df.loc[idx, 'classID']
        aud = AudioUtil.open(audio_file)
        reaud = AudioUtil.resample(aud, self.sr)
        rechan = AudioUtil.rechannel(reaud, self.channel)
        dur_aud = AudioUtil.pad_trunc(rechan, self.duration)
        shift_aud = AudioUtil.time_shift(dur_aud, self.shift_pct)
        sgram = AudioUtil.spectro_gram(shift_aud, n_mels=64, n_fft=1024, hop_len=None)
        aug_sgram = AudioUtil.spectro_augment(sgram, max_mask_pct=0.1, n_freq_masks=2, n_time_masks=2)
        return aug_sgram, class_id


In [9]:
from torch.utils.data import random_split
from torch.nn import init
import torch.nn as nn

In [10]:
data_path = "/content/urbansound8k"
myds = SoundDS(df, data_path)
num_items = len(myds)
num_train = round(num_items * 0.85)
num_val = num_items - num_train
train_ds, val_ds = random_split(myds, [num_train, num_val])

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl = torch.utils.data.DataLoader(val_ds, batch_size=16, shuffle=False)

In [11]:
class AudioClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        conv_layers = []

        #1
        self.conv1 = nn.Conv2d(2, 8, kernel_size=(3,3), stride=(2, 2), padding=(2, 2))
        self.relu1 = nn.ReLU()
        self.bn1 = nn.BatchNorm2d(8)
        init.kaiming_normal_(self.conv1.weight, a=0.1)
        self.conv1.bias.data.zero_()
        conv_layers += [self.conv1, self.relu1, self.bn1]

        #2
        self.conv2 = nn.Conv2d(8, 16, kernel_size=(3,3), stride=(2, 2), padding=(1, 1))
        self.relu2 = nn.ReLU()
        self.bn2 = nn.BatchNorm2d(16)
        init.kaiming_normal_(self.conv2.weight, a=0.1)
        self.conv2.bias.data.zero_()
        conv_layers += [self.conv2, self.relu2, self.bn2]

        #3
        self.conv3 = nn.Conv2d(16, 32, kernel_size=(3,3), stride=(2, 2), padding=(1, 1))
        self.relu3 = nn.ReLU()
        self.bn3 = nn.BatchNorm2d(32)
        init.kaiming_normal_(self.conv3.weight, a=0.1)
        self.conv3.bias.data.zero_()
        conv_layers += [self.conv3, self.relu3, self.bn3]

        #4
        self.conv4 = nn.Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        self.relu4 = nn.ReLU()
        self.bn4 = nn.BatchNorm2d(64)
        init.kaiming_normal_(self.conv4.weight, a=0.1)
        self.conv4.bias.data.zero_()
        conv_layers += [self.conv4, self.relu4, self.bn4]

        self.ap = nn.AdaptiveAvgPool2d(output_size=1)
        self.lin = nn.Linear(in_features=64, out_features=10)

        self.conv = nn.Sequential(*conv_layers)

    def forward(self, x):
        x = self.conv(x)

        x = self.ap(x)
        x = x.view(x.shape[0], -1)

        x = self.lin(x)

        return x




In [13]:
class MiniVGGClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=2, out_channels=16, kernel_size=(3, 3), stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(16),
            nn.Conv2d(in_channels=16, out_channels=16, kernel_size=(3, 3), stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(16),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=(3, 3), stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.block4 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=(3, 3), stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=(3, 3), stride=1, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.avgpool = nn.AdaptiveAvgPool2d(output_size=1)
        self.lin = nn.Sequential(
            nn.Linear(in_features=128, out_features=128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(in_features=128, out_features=10)
        )
        self._init_weights()


    def _init_weights(self):
      for m in self.modules():
          if isinstance(m, nn.Conv2d):
              init.kaiming_normal_(m.weight, a=0.1)
              if m.bias is not None:
                  m.bias.data.zero_()
          elif isinstance(m, nn.Linear):
              init.kaiming_normal_(m.weight, a=0.1)
              if m.bias is not None:
                  m.bias.data.zero_()
    def forward(self, x):
      x = self.block1(x)
      x = self.block2(x)
      x = self.block3(x)
      x = self.block4(x)

      x = self.avgpool(x)
      x = x.view(x.shape[0], -1)
      x = self.lin(x)
      return x



In [14]:
myModel = MiniVGGClassifier()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
myModel = myModel.to(device)
next(myModel.parameters()).device

cuda:0


device(type='cuda', index=0)

In [15]:
def training(model, train_dl, num_epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.001,
                                                steps_per_epoch=int(len(train_dl)),
                                                epochs=num_epochs,
                                                anneal_strategy='linear')
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct_prediction = 0
        total_prediction = 0

        for i, data in enumerate(train_dl):
            inputs, labels = data[0].to(device), data[1].to(device)

            inputs_m, inputs_s = inputs.mean(), inputs.std()
            inputs = (inputs - inputs_m) / inputs_s

            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()

            running_loss += loss.item()

            _, prediction = torch.max(outputs, 1)
            correct_prediction += (prediction == labels).sum().item()
            total_prediction += prediction.shape[0]
            num_batches = len(train_dl)
            avg_loss = running_loss / num_batches
            acc = correct_prediction/total_prediction
            print(f'Epoch: {epoch}, Loss: {avg_loss:.7f}, Accuracy: {acc:7f}')

        print('Finished Training')


In [ ]:
num_epochs=30
training(myModel, train_dl, num_epochs)

Выходные данные были обрезаны до нескольких последних строк (5000).
Epoch: 19, Loss: 0.0523488, Accuracy: 0.929418
Epoch: 19, Loss: 0.0524134, Accuracy: 0.930021
Epoch: 19, Loss: 0.0525721, Accuracy: 0.930085
Epoch: 19, Loss: 0.0537157, Accuracy: 0.929097
Epoch: 19, Loss: 0.0538457, Accuracy: 0.929688
Epoch: 19, Loss: 0.0546171, Accuracy: 0.928719
Epoch: 19, Loss: 0.0550001, Accuracy: 0.928279
Epoch: 19, Loss: 0.0551320, Accuracy: 0.928862
Epoch: 19, Loss: 0.0558813, Accuracy: 0.928427
Epoch: 19, Loss: 0.0564865, Accuracy: 0.928000
Epoch: 19, Loss: 0.0574561, Accuracy: 0.927083
Epoch: 19, Loss: 0.0577683, Accuracy: 0.927165
Epoch: 19, Loss: 0.0586376, Accuracy: 0.926758
Epoch: 19, Loss: 0.0594387, Accuracy: 0.925872
Epoch: 19, Loss: 0.0597159, Accuracy: 0.925962
Epoch: 19, Loss: 0.0597518, Accuracy: 0.926527
Epoch: 19, Loss: 0.0606619, Accuracy: 0.925189
Epoch: 19, Loss: 0.0613155, Accuracy: 0.924812
Epoch: 19, Loss: 0.0617116, Accuracy: 0.924440
Epoch: 19, Loss: 0.0620784, Accuracy: 0

In [ ]:
def inference (model, val_dl):
  correct_prediction = 0
  total_prediction = 0

  with torch.no_grad():
    for data in val_dl:
      inputs, labels = data[0].to(device), data[1].to(device)

      inputs_m, inputs_s = inputs.mean(), inputs.std()
      inputs = (inputs - inputs_m) / inputs_s

      outputs = model(inputs)

      _, prediction = torch.max(outputs,1)
      correct_prediction += (prediction == labels).sum().item()
      total_prediction += prediction.shape[0]

  acc = correct_prediction/total_prediction
  print(f'Accuracy: {acc:.2f}, Total items: {total_prediction}')

inference(myModel, val_dl)

Accuracy: 0.94, Total items: 1310


In [18]:
torch.save(myModel.state_dict(), "/content/model2.pt")